
# Phase 2: Unsupervised Learning

## Gaussian Mixture Model (GMM)

### Objective

The objective of this phase is to identify meaningful customer segments based on their purchasing behavior, engagement, and loyalty characteristics using unsupervised learning.

Gaussian Mixture Model (GMM) is selected as an alternative to K-Means because it provides probabilistic cluster assignments and can model clusters with different covariance structures.

The resulting customer segments will be evaluated, interpreted, and used to support personalized customer retention recommendations in the CCO Retail Intelligence and Customer Retention Advisory System.



## Data Preparation

The dataset is prepared for unsupervised clustering by excluding the target variable (`churn_flag`), the customer identifier (`customer_id`), and the potentially leakage-prone feature (`churn_risk`).

Categorical features will be encoded using One-Hot Encoding, while numerical features will be standardized using StandardScaler.

These preprocessing steps help prepare the feature matrix for GMM clustering and support a fair comparison with K-Means.


In [ ]:

import pandas as pd
from pathlib import Path

# Locate the original dataset
possible_paths = [
    Path("Dataset/retail_customer_churn.csv"),
    Path("../Dataset/retail_customer_churn.csv")
]

dataset_path = next(
    (path for path in possible_paths if path.exists()),
    None
)

if dataset_path is None:
    raise FileNotFoundError("Dataset file not found. Check the project folder.")

# Load the dataset
df = pd.read_csv(dataset_path)

# Display dataset dimensions and first five rows
print("Dataset shape:", df.shape)
display(df.head())


Dataset shape: (100000, 18)


,customer_id,age_group,gender,region,customer_segment,preferred_channel,purchase_frequency,avg_order_value,total_spent,recency_days,website_visits,discount_usage_rate,email_open_rate,cart_abandonment_rate,loyalty_score,engagement_score,churn_risk,churn_flag
0,C000001,18-24,Other,North,Loyal,Online,5,381.27,2108.80,189,27,0.38,0.13,0.42,33,35.5,Medium,1
1,C000002,25-34,Other,South,New,Mobile App,12,376.65,3843.06,277,68,0.51,0.06,0.07,68,49.5,Medium,1
2,C000003,55+,Other,North,VIP,Online,5,37.73,224.65,271,23,0.64,0.22,0.46,73,38.0,High,1
3,C000004,18-24,Male,North,Returning,Mobile App,12,191.19,1960.79,89,93,0.66,0.31,0.13,93,59.0,Low,0
4,C000005,55+,Other,North,Returning,Mobile App,2,279.64,594.79,264,4,0.73,0.27,0.58,11,34.5,High,1


In [2]:

# Remove identifier, target, and churn risk columns
columns_to_drop = ["customer_id", "churn_flag", "churn_risk"]

X = df.drop(columns=columns_to_drop)

print("Original dataset shape:", df.shape)
print("Clustering features shape:", X.shape)
print("\nMissing values:")
print(X.isnull().sum())


Original dataset shape: (100000, 18)
Clustering features shape: (100000, 15)

Missing values:
age_group                0
gender                   0
region                   0
customer_segment         0
preferred_channel        0
purchase_frequency       0
avg_order_value          0
total_spent              0
recency_days             0
website_visits           0
discount_usage_rate      0
email_open_rate          0
cart_abandonment_rate    0
loyalty_score            0
engagement_score         0
dtype: int64



### Feature Encoding and Scaling

Categorical features are transformed using One-Hot Encoding to represent categories numerically without imposing an artificial order.

Numerical features are standardized using StandardScaler to prevent features with larger numerical scales from dominating the clustering process.

The resulting feature matrix is used as input to the Gaussian Mixture Model (GMM).


In [4]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify numerical and categorical features
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "str", "category"]).columns.tolist()

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

# Define preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ]
)

# Apply encoding and scaling
X_processed = preprocessor.fit_transform(X)

print("\nOriginal feature shape:", X.shape)
print("Processed feature shape:", X_processed.shape)


Numerical features: ['purchase_frequency', 'avg_order_value', 'total_spent', 'recency_days', 'website_visits', 'discount_usage_rate', 'email_open_rate', 'cart_abandonment_rate', 'loyalty_score', 'engagement_score']
Categorical features: ['age_group', 'gender', 'region', 'customer_segment', 'preferred_channel']

Original feature shape: (100000, 15)
Processed feature shape: (100000, 30)
